# 02 - Artificial-gap validation protocol

Loads the canonical artificial-gap pool and explains the validation
protocol. Fully executable on the public data included in this repository.

## Why artificial gaps?

Real gaps in the record (sensor outages, etc.) have no withheld ground
truth -- the sensor was genuinely offline, so there's nothing to compare a
reconstruction against. To get quantitative evidence about reconstruction
accuracy, we instead pick stretches of the record that ARE observed,
deliberately hide the true values for a block of consecutive days, run a
candidate method as if those days were missing, and score its predictions
against the secretly retained true values.

## Leakage prevention, in plain terms

"Leakage" means a method accidentally seeing information it shouldn't have
-- most importantly, the true value of a day that's supposed to be hidden.
We prevent this by: (1) masking only the target column, never the external
predictor data, for an artificial gap; (2) ensuring any feature computed
from the target's own history (e.g. "value 3 days ago") becomes missing if
it would reach into the hidden gap; (3) scoring only on the secretly
retained true values of hidden days, never on visible days.


In [ ]:
import pandas as pd
import sys
sys.path.insert(0, "../src")
from coastal_gap_reconstruction.data_loading import load_validation_gap_pool

gap_pool = load_validation_gap_pool("../data_public/chlorophyll/chlorophyll_validation_gaps.csv")
print(f"Total artificial gaps in canonical pool: {len(gap_pool)}")
gap_pool.head()


## Gap-length distribution

In [ ]:
length_counts = gap_pool["gap_length"].value_counts().sort_index()
length_counts


In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(7, 4))
length_counts.plot(kind="bar", ax=ax, color="steelblue")
ax.set_xlabel("Gap length (days)")
ax.set_ylabel("Number of artificial gaps")
ax.set_title("Canonical artificial-gap pool: gap-length distribution")
plt.tight_layout()
plt.show()


## Season and event-status breakdown

In [ ]:
print(gap_pool["season"].value_counts())
print()
print(gap_pool["is_high_chl_event"].value_counts())


## Maximum validated gap length

The canonical pool covers gap lengths up to 60 days. Any reconstruction
claim for a longer gap (e.g. the 256-day real gap in this record) is
extrapolation beyond validated evidence -- see `docs/evidence_hierarchy.md`.
